In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DREAM POLYP UNIFIED NET — v3.0  (Publication Grade)                      ║
║   Architecture: SE+PR-CNN ➔ UNet(Encoder) ─┬─> UNet(Decoder) ➔ Mask      ║
║                                             └─> PD-CNN ➔ PCC ➔ CLF        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  ROOT CAUSE ANALYSIS — Why Val_Dice was stuck at 0.41:                     ║
║                                                                              ║
║  1. WRONG LOSS: Plain Dice treats all pixels equally. Polyps occupy         ║
║     ~5-10% of pixels → model learns blob shape, never sharp boundaries.     ║
║     FIX: Focal Tversky Loss (alpha=0.7 penalizes FN = missed polyp).        ║
║                                                                              ║
║  2. CLASS IMBALANCE IGNORED: Standard BCE weights all pixels equally.       ║
║     95% are background → model biased toward predicting zeros.              ║
║     FIX: Weighted BCE with pos_weight=10 for polyp pixels.                 ║
║                                                                              ║
║  3. NO EDGE SUPERVISION: Model had no direct signal to learn boundaries.    ║
║     FIX: Sobel boundary loss added as third component.                      ║
║                                                                              ║
║  4. CLF GRADIENT POLLUTION: Classification converged at epoch ~30.          ║
║     Its gradients then corrupt shared encoder weights.                      ║
║     FIX: Phase-aware loss weighting — auto-switch when acc > 0.88.         ║
║                                                                              ║
║  5. DEEP SUPERVISION MISSING: Mid-decoder layers had no direct gradient.    ║
║     FIX: Auxiliary mask heads at d4(32²), d3(64²), d2(128²) decoder levels.║
║                                                                              ║
║  6. LR DECAYED TO ~0 BY EPOCH 79: Cosine schedule had no floor.            ║
║     FIX: min_lr = 5e-5 clamp prevents gradient starvation.                 ║
║                                                                              ║
║  7. VALIDATION WAS ZIPPED: Shorter dataset truncated longer one.            ║
║     FIX: Uncoupled seg and clf validation — each runs independently.        ║
║                                                                              ║
║  8. AUGMENTATION TOO WEAK: No rotation, no zoom → poor generalization.      ║
║     FIX: 90° rotations + random zoom-crop + saturation/hue jitter.         ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os
import glob
import gc
import numpy as np

# ── Set BEFORE importing TensorFlow ──────────────────────────────────────────
os.environ["TF_GPU_ALLOCATOR"]   = "cuda_malloc_async"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow.keras import layers, models, Input, mixed_precision
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# GPU SETUP
# ─────────────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU Memory Growth Enabled — {len(gpus)} GPU(s) found")
    except RuntimeError as e:
        print(e)

# Cap thread pools — prevents AUTOTUNE over-allocating pinned host RAM
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
IMG_SIZE      = 256
BATCH_SIZE    = 4        # RTX 4060 Laptop (5.5 GB VRAM) — verified safe
EPOCHS        = 120      # Extra headroom; early stopping guards against waste
WARMUP_EPOCHS = 3
PREFETCH      = 2        # Hard cap — no AUTOTUNE (causes 4 GB pinned RAM OOM)
NUM_WORKERS   = 4

# Phase-switching: once clf converges, redirect gradient budget to segmentation
CLF_CONVERGE_ACC    = 0.88   # val accuracy threshold
CLF_CONVERGE_EPOCHS = 3      # must hold for this many consecutive epochs
SEG_W_PHASE1 = 0.80          # seg weight during phase 1
CLF_W_PHASE1 = 0.20          # clf weight during phase 1
SEG_W_PHASE2 = 0.97          # seg weight after clf convergence
CLF_W_PHASE2 = 0.03          # minimal — keeps clf from catastrophic forgetting

# Deep supervision aux loss weights (lower than primary — they guide, not dominate)
AUX_W_D4 = 0.20
AUX_W_D3 = 0.15
AUX_W_D2 = 0.10

DATA_PATH    = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2'
POSITIVE_DIR = os.path.join(DATA_PATH, "imagesAll_positive")
NEGATIVE_DIR = os.path.join(DATA_PATH, "sequenceData", "negativeOnly")
SAVE_PATH    = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/unet-trsenet/depth-se-polypgen/unified_model_v3.keras'

mixed_precision.set_global_policy('mixed_float16')
print("✓ Mixed Precision (float16) enabled")


# ─────────────────────────────────────────────────────────────────────────────
# 📊  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
def load_segmentation_paths(base_path):
    all_images, all_masks = [], []
    for i in range(1, 7):
        cid      = f"C{i}"
        img_dir  = os.path.join(base_path, f"data_{cid}", f"images_{cid}")
        mask_dir = os.path.join(base_path, f"data_{cid}", f"masks_{cid}")
        if not (os.path.exists(img_dir) and os.path.exists(mask_dir)):
            continue
        img_paths = []
        for ext in ("*.jpg", "*.JPG", "*.png", "*.jpeg"):
            img_paths.extend(glob.glob(os.path.join(img_dir, ext)))
        for img_p in img_paths:
            fname  = os.path.basename(img_p)
            name, ext = os.path.splitext(fname)
            mask_p = os.path.join(mask_dir, f"{name}_mask{ext}")
            if os.path.exists(mask_p):
                all_images.append(img_p)
                all_masks.append(mask_p)
    print(f"  Segmentation samples: {len(all_images)}")
    return all_images, all_masks


def load_classification_paths():
    valid_ext = ('*.png','*.jpg','*.jpeg','*.bmp','*.tif','*.PNG','*.JPG','*.JPEG')
    pos_files, neg_files = [], []
    for ext in valid_ext:
        pos_files.extend(glob.glob(os.path.join(POSITIVE_DIR, "**", ext), recursive=True))
    for i in range(1, 14):
        folder = os.path.join(NEGATIVE_DIR, f"seq{i}_neg")
        if os.path.exists(folder):
            for ext in valid_ext:
                neg_files.extend(glob.glob(os.path.join(folder, "**", ext), recursive=True))
    file_paths = pos_files + neg_files
    labels     = [1.0] * len(pos_files) + [0.0] * len(neg_files)
    print(f"  Classification samples: pos={len(pos_files)}, neg={len(neg_files)}")
    return np.array(file_paths), np.array(labels, dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# 🗄️  PARSERS & AUGMENTATION
# ─────────────────────────────────────────────────────────────────────────────
def augment(img, mask):
    """
    v3.0 strong paired augmentation.
    All spatial ops applied identically to img & mask.
    Colour ops applied to img only (mask is binary — colour is meaningless).
    """
    # ── Paired spatial: flips ────────────────────────────────────────────────
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)

    # ── Paired spatial: 90° rotation ─────────────────────────────────────────
    # Polyps appear at all orientations in colonoscopy — rotation is critical
    k    = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img  = tf.image.rot90(img,  k)
    mask = tf.image.rot90(mask, k)

    # ── Paired spatial: random zoom-crop ─────────────────────────────────────
    # Stack img+mask → crop together → split → resize back to IMG_SIZE
    # This preserves the exact img↔mask spatial correspondence
    if tf.random.uniform(()) > 0.4:
        scale   = tf.random.uniform((), minval=0.78, maxval=1.0)
        crop_sz = tf.cast(tf.cast(IMG_SIZE, tf.float32) * scale, tf.int32)
        stacked = tf.concat([img, mask], axis=-1)           # [H, W, 4]
        stacked = tf.image.random_crop(stacked, [crop_sz, crop_sz, 4])
        img     = tf.image.resize(stacked[:, :, :3], [IMG_SIZE, IMG_SIZE])
        mask    = tf.image.resize(stacked[:, :, 3:], [IMG_SIZE, IMG_SIZE],
                                  method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
        mask    = tf.where(mask > 0.5, 1.0, 0.0)

    # ── Image-only: colour jitter ─────────────────────────────────────────────
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.80, upper=1.20)
    img = tf.image.random_saturation(img, lower=0.75, upper=1.25)
    img = tf.image.random_hue(img, max_delta=0.05)
    img = tf.clip_by_value(img, 0.0, 1.0)

    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, mask


def parse_seg_element(img_path, mask_path):
    img  = tf.io.read_file(img_path)
    img  = tf.image.decode_image(img, channels=3, expand_animations=False)
    img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img  = tf.cast(img, tf.float32) / 255.0
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE],
                           method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    mask = tf.cast(mask, tf.float32) / 255.0
    mask = tf.where(mask > 0.5, 1.0, 0.0)
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, mask


def parse_clf_element(img_path, label):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, tf.reshape(label, [1])


def build_datasets(s_tr_x, s_tr_y, s_val_x, s_val_y,
                   c_tr_x, c_tr_y, c_val_x, c_val_y,
                   steps_per_epoch):
    """
    v3.0 pipeline:
    - No .cache() — avoids pinned RAM OOM on laptop GPUs
    - prefetch=PREFETCH=2 — hard cap, not AUTOTUNE
    - Validation datasets are INDEPENDENT (not zipped) — each runs to completion
    - steps_per_epoch enforced via .take() — guarantees identical epoch lengths
    """
    train_seg_ds = (
        tf.data.Dataset.from_tensor_slices((s_tr_x, s_tr_y))
        .shuffle(min(len(s_tr_x), 1000), reshuffle_each_iteration=True)
        .map(parse_seg_element, num_parallel_calls=NUM_WORKERS)
        .map(augment,           num_parallel_calls=NUM_WORKERS)
        .batch(BATCH_SIZE, drop_remainder=True)
        .take(steps_per_epoch)
        .prefetch(PREFETCH)
    )
    train_clf_ds = (
        tf.data.Dataset.from_tensor_slices((c_tr_x, c_tr_y))
        .shuffle(min(len(c_tr_x), 1000), reshuffle_each_iteration=True)
        .map(parse_clf_element, num_parallel_calls=NUM_WORKERS)
        .batch(BATCH_SIZE, drop_remainder=True)
        .take(steps_per_epoch)
        .prefetch(PREFETCH)
    )
    # Validation: fully independent, no zipping, runs to completion
    val_seg_ds = (
        tf.data.Dataset.from_tensor_slices((s_val_x, s_val_y))
        .map(parse_seg_element, num_parallel_calls=NUM_WORKERS)
        .batch(BATCH_SIZE)
        .prefetch(PREFETCH)
    )
    val_clf_ds = (
        tf.data.Dataset.from_tensor_slices((c_val_x, c_val_y))
        .map(parse_clf_element, num_parallel_calls=NUM_WORKERS)
        .batch(BATCH_SIZE)
        .prefetch(PREFETCH)
    )
    return train_seg_ds, train_clf_ds, val_seg_ds, val_clf_ds


# ─────────────────────────────────────────────────────────────────────────────
# 📐  ARCHITECTURAL BLOCKS  ← UNCHANGED (architecture is not the problem)
# ─────────────────────────────────────────────────────────────────────────────
def se_block(x, ratio=8):
    c = x.shape[-1]
    s = layers.GlobalAveragePooling2D()(x)
    e = layers.Dense(max(c // ratio, 1), activation='relu', use_bias=False)(s)
    e = layers.Dense(c, activation='sigmoid', use_bias=False)(e)
    e = layers.Reshape((1, 1, c))(e)
    return layers.multiply([x, e])


def se_pr_cnn_stem(inputs, filters=32):
    """SE + Pointwise-Residual CNN — preprocessing stem before UNet encoder."""
    x   = layers.Conv2D(filters, 3, padding='same', use_bias=False)(inputs)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    res = layers.Conv2D(filters, 1, padding='same', use_bias=False)(inputs)
    res = layers.BatchNormalization()(res)
    x   = layers.add([x, res])
    return se_block(x)


def ds_conv_block(x, filters, dropout_rate=0.0):
    """Multi-scale depthwise-separable block with SE and residual."""
    f4 = filters // 4

    b1 = layers.SeparableConv2D(f4, 3, padding='same', use_bias=False)(x)
    b1 = layers.BatchNormalization()(b1); b1 = layers.Activation('relu')(b1)

    b2 = layers.SeparableConv2D(f4, 5, padding='same', use_bias=False)(x)
    b2 = layers.BatchNormalization()(b2); b2 = layers.Activation('relu')(b2)

    # Dilated 3×3 replaces 7×7 — same receptive field, 4× fewer FLOPs
    b3 = layers.SeparableConv2D(f4, 3, dilation_rate=3, padding='same', use_bias=False)(x)
    b3 = layers.BatchNormalization()(b3); b3 = layers.Activation('relu')(b3)

    b4 = layers.SeparableConv2D(f4, 1, padding='same', use_bias=False)(x)
    b4 = layers.BatchNormalization()(b4); b4 = layers.Activation('relu')(b4)

    merged = layers.Concatenate()([b1, b2, b3, b4])
    merged = se_block(merged)

    if x.shape[-1] == filters:
        merged = layers.add([x, merged])
    if dropout_rate > 0:
        merged = layers.SpatialDropout2D(dropout_rate)(merged)
    return merged


def pd_cnn_head(bottleneck, filters=128):
    """Polyp-Discriminative CNN branch — sits on the bottleneck for classification."""
    x = layers.SeparableConv2D(filters, 3, padding='same', use_bias=False)(bottleneck)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = se_block(x)
    x = layers.SeparableConv2D(filters // 2, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    return x


@tf.keras.utils.register_keras_serializable()
class PCCLayer(layers.Layer):
    """
    Compact Pearson Correlation Coefficient feature transformation.
    Runs in float32 for numerical stability regardless of global policy.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        x    = tf.cast(inputs, tf.float32)
        half = tf.shape(x)[-1] // 2
        a, b = x[:, :half], x[:, half:]
        eps  = 1e-8
        a_c  = a - tf.reduce_mean(a, axis=-1, keepdims=True)
        b_c  = b - tf.reduce_mean(b, axis=-1, keepdims=True)
        rho  = (tf.reduce_sum(a_c * b_c, axis=-1, keepdims=True) /
                (tf.norm(a_c, axis=-1, keepdims=True) *
                 tf.norm(b_c, axis=-1, keepdims=True) + eps))
        a_n  = a_c / (tf.norm(a_c, axis=-1, keepdims=True) + eps)
        b_n  = b_c / (tf.norm(b_c, axis=-1, keepdims=True) + eps)
        return tf.concat([a_n, b_n, rho], axis=-1)

    def get_config(self):
        return super().get_config()


# ─────────────────────────────────────────────────────────────────────────────
# 🏗️  UNIFIED MULTI-TASK MODEL
#
#  Input ➔ SE+PR-CNN ➔ UNet(Encoder) ─┬─> UNet(Decoder) ➔ mask_output
#                                      │    └> aux_d4, aux_d3, aux_d2 (deep supervision)
#                                      └─> PD-CNN ➔ PCC ➔ Standardization ➔ class_output
#
#  v3.0 ADDITIONS (training only — inference path unchanged):
#    • aux_d4: 1×1 conv on d4(32²)  → upsample ×8  → 256² sigmoid
#    • aux_d3: 1×1 conv on d3(64²)  → upsample ×4  → 256² sigmoid
#    • aux_d2: 1×1 conv on d2(128²) → upsample ×2  → 256² sigmoid
# ─────────────────────────────────────────────────────────────────────────────
def build_unified_model():
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")

    # ── SE+PR-CNN Stem ────────────────────────────────────────────────────────
    stem = se_pr_cnn_stem(inputs, filters=32)   # 256×256×32

    # ── UNet Encoder ─────────────────────────────────────────────────────────
    c1 = ds_conv_block(stem, 32);           p1 = layers.MaxPooling2D(2)(c1)  # 128
    c2 = ds_conv_block(p1,   64);           p2 = layers.MaxPooling2D(2)(c2)  # 64
    c3 = ds_conv_block(p2,  128, 0.1);      p3 = layers.MaxPooling2D(2)(c3)  # 32
    c4 = ds_conv_block(p3,  256, 0.2);      p4 = layers.MaxPooling2D(2)(c4)  # 16

    # ── Bottleneck ────────────────────────────────────────────────────────────
    bottleneck = ds_conv_block(p4, 512, 0.3)   # 16×16×512

    # ══════════════════════════════════════════════════════════════════════════
    # BRANCH A — UNet Decoder → Segmentation
    # ══════════════════════════════════════════════════════════════════════════
    u4 = layers.Conv2DTranspose(256, 2, strides=2, padding='same')(bottleneck) # 32
    d4 = ds_conv_block(layers.concatenate([u4, c4]), 256, 0.2)

    u3 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(d4)         # 64
    d3 = ds_conv_block(layers.concatenate([u3, c3]), 128, 0.1)

    u2 = layers.Conv2DTranspose(64,  2, strides=2, padding='same')(d3)         # 128
    d2 = ds_conv_block(layers.concatenate([u2, c2]), 64)

    u1 = layers.Conv2DTranspose(32,  2, strides=2, padding='same')(d2)         # 256
    d1 = ds_conv_block(layers.concatenate([u1, c1]), 32)

    # Primary segmentation output (float32 for stability)
    mask_output = layers.Conv2D(
        1, 1, activation='sigmoid', dtype='float32', name='mask_output'
    )(d1)

    # ── Deep Supervision Auxiliary Heads ──────────────────────────────────────
    # Each produces a full-resolution sigmoid prediction from a mid-decoder level.
    # This gives the decoder direct gradient signal at every spatial scale,
    # breaking the "blob plateau" where only the final layer gets supervised.
    #
    # d4: 32×32  → ×8  upsample → 256×256
    aux_d4 = layers.Conv2D(1, 1, use_bias=False)(d4)
    aux_d4 = layers.UpSampling2D(size=(8, 8), interpolation='bilinear')(aux_d4)
    aux_d4 = layers.Activation('sigmoid', dtype='float32', name='aux_d4')(aux_d4)

    # d3: 64×64  → ×4  upsample → 256×256
    aux_d3 = layers.Conv2D(1, 1, use_bias=False)(d3)
    aux_d3 = layers.UpSampling2D(size=(4, 4), interpolation='bilinear')(aux_d3)
    aux_d3 = layers.Activation('sigmoid', dtype='float32', name='aux_d3')(aux_d3)

    # d2: 128×128 → ×2  upsample → 256×256
    aux_d2 = layers.Conv2D(1, 1, use_bias=False)(d2)
    aux_d2 = layers.UpSampling2D(size=(2, 2), interpolation='bilinear')(aux_d2)
    aux_d2 = layers.Activation('sigmoid', dtype='float32', name='aux_d2')(aux_d2)

    # ══════════════════════════════════════════════════════════════════════════
    # BRANCH B — PD-CNN ➔ PCC ➔ Standardization ➔ Classification
    # ══════════════════════════════════════════════════════════════════════════
    pd_feat  = pd_cnn_head(bottleneck, filters=128)
    gap      = layers.GlobalAveragePooling2D()(pd_feat)            # [B, 128]
    pcc      = PCCLayer(name='pcc')(gap)                           # [B, 129]
    pcc_f32  = layers.Lambda(
        lambda t: tf.cast(t, tf.float32), name='pcc_cast'
    )(pcc)
    std      = layers.BatchNormalization(dtype='float32')(pcc_f32) # Z-score

    fc1      = layers.Dense(128, activation='relu', dtype='float32')(std)
    fc1      = layers.Dropout(0.4)(fc1)
    fc2      = layers.Dense(64,  activation='relu', dtype='float32')(fc1)
    fc2      = layers.Dropout(0.3)(fc2)
    clf_out  = layers.Dense(1, activation='sigmoid', dtype='float32',
                            name='class_output')(fc2)

    # 5 outputs: primary mask, 3 aux masks, classification
    return models.Model(
        inputs=inputs,
        outputs=[mask_output, aux_d4, aux_d3, aux_d2, clf_out],
        name="Dream_Polyp_Unified_Net_v3"
    )


# ─────────────────────────────────────────────────────────────────────────────
# 🧮  LOSS FUNCTIONS — v3.0 (the core fix for Dice plateau)
# ─────────────────────────────────────────────────────────────────────────────
def dice_coef(y_true, y_pred, smooth=1.0):
    """Dice coefficient — used as METRIC only, not for training."""
    y_t   = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_p   = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    inter = tf.reduce_sum(y_t * y_p)
    return (2.0 * inter + smooth) / (tf.reduce_sum(y_t) + tf.reduce_sum(y_p) + smooth)


def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75, smooth=1.0):
    """
    Focal Tversky Loss — best loss for small, imbalanced medical segmentation.

    Tversky index = TP / (TP + alpha*FN + beta*FP)
      alpha=0.7: heavily penalizes False Negatives (missing polyp pixels)
      beta=0.3:  tolerates some False Positives (over-segmentation)
    Focal exponent (1/gamma): concentrates gradient on hard/uncertain pixels.

    Why this beats plain Dice for PolypGen:
      - Polyps are small (5-10% of pixels) → FN dominates standard Dice
      - alpha weighting forces the model to chase the polyp, not background
      - Focal term ensures the model doesn't ignore hard boundary pixels
    """
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_p = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    tp  = tf.reduce_sum(y_t * y_p)
    fp  = tf.reduce_sum((1.0 - y_t) * y_p)
    fn  = tf.reduce_sum(y_t * (1.0 - y_p))
    tversky = (tp + smooth) / (tp + alpha * fn + beta * fp + smooth)
    return tf.pow(1.0 - tversky, 1.0 / gamma)


def weighted_bce_loss(y_true, y_pred, pos_weight=10.0):
    """
    Weighted Binary Cross-Entropy.
    pos_weight=10: polyp pixels contribute 10× more to the loss than background.
    Directly corrects the ~90:10 class imbalance in PolypGen.
    """
    y_t  = tf.cast(y_true, tf.float32)
    y_p  = tf.cast(y_pred, tf.float32)
    bce  = tf.keras.losses.binary_crossentropy(y_t, y_p)  # [B, H, W]
    wmap = y_t * (pos_weight - 1.0) + 1.0                 # [B, H, W, 1]
    if len(wmap.shape) > len(bce.shape):
        wmap = tf.squeeze(wmap, axis=-1)
    return tf.reduce_mean(bce * wmap)


def boundary_loss(y_true, y_pred):
    """
    Sobel Edge Boundary Loss.
    Computes gradient magnitude on both GT mask and prediction,
    penalises mismatch at boundary pixels.
    Forces the model beyond blob-level predictions to precise boundaries.
    This is the key fix for the Dice 0.41 plateau.
    """
    y_t     = tf.cast(y_true, tf.float32)
    y_p     = tf.cast(y_pred, tf.float32)
    edges_t = tf.image.sobel_edges(y_t)  # [B, H, W, C, 2]
    edges_p = tf.image.sobel_edges(y_p)
    mag_t   = tf.sqrt(tf.reduce_sum(tf.square(edges_t), axis=-1) + 1e-8)
    mag_p   = tf.sqrt(tf.reduce_sum(tf.square(edges_p), axis=-1) + 1e-8)
    return tf.reduce_mean(tf.abs(mag_t - mag_p))


def combined_seg_loss(y_true, y_pred):
    """
    Combined segmentation loss = 0.5×FTL + 0.3×WBCE + 0.2×Boundary

    FTL (0.5):      Primary driver — recall-focused, handles class imbalance
    WBCE (0.3):     Pixel-level calibration with pos_weight=10
    Boundary (0.2): Edge precision — breaks the blob plateau
    """
    y_t  = tf.cast(y_true, tf.float32)
    y_p  = tf.cast(y_pred, tf.float32)
    ftl  = focal_tversky_loss(y_t, y_p)
    wbce = weighted_bce_loss(y_t, y_p)
    bnd  = boundary_loss(y_t, y_p)
    return 0.5 * ftl + 0.3 * wbce + 0.2 * bnd


# ─────────────────────────────────────────────────────────────────────────────
# 🧠  UNIFIED TRAINER
# ─────────────────────────────────────────────────────────────────────────────
class UnifiedTrainer(models.Model):
    def __init__(self, unified_model, **kwargs):
        super().__init__(**kwargs)
        self.unified_model = unified_model
        self.bce_loss      = tf.keras.losses.BinaryCrossentropy()

        # Phase-switchable loss weights (tf.Variable → no retracing needed)
        self.seg_w = tf.Variable(SEG_W_PHASE1, trainable=False, dtype=tf.float32)
        self.clf_w = tf.Variable(CLF_W_PHASE1, trainable=False, dtype=tf.float32)

        # Train metrics
        self.m_loss_seg = tf.keras.metrics.Mean(name="loss_seg")
        self.m_loss_clf = tf.keras.metrics.Mean(name="loss_clf")
        self.m_dice     = tf.keras.metrics.Mean(name="dice")
        self.m_clf_acc  = tf.keras.metrics.BinaryAccuracy(name="clf_acc")

        # Validation metrics — separate accumulators, independent of train
        self.v_loss_seg = tf.keras.metrics.Mean(name="val_loss_seg")
        self.v_loss_clf = tf.keras.metrics.Mean(name="val_loss_clf")
        self.v_dice     = tf.keras.metrics.Mean(name="val_dice")
        self.v_clf_acc  = tf.keras.metrics.BinaryAccuracy(name="val_clf_acc")

    def compile(self, optimizer, seg_loss_fn=combined_seg_loss):
        super().compile()
        self.optimizer   = optimizer
        self.seg_loss_fn = seg_loss_fn

    @property
    def metrics(self):
        return [self.m_loss_seg, self.m_loss_clf, self.m_dice, self.m_clf_acc,
                self.v_loss_seg, self.v_loss_clf, self.v_dice, self.v_clf_acc]

    @tf.function
    def train_step(self, seg_batch, clf_batch):
        """
        Single GradientTape combining seg + clf in one forward+backward pass.
        Deep supervision: model returns [mask, aux_d4, aux_d3, aux_d2, class]
        """
        s_imgs, s_masks = seg_batch
        c_imgs, c_lbls  = clf_batch

        with tf.GradientTape() as tape:
            # Forward pass on seg batch → all 5 outputs
            pred_mask, pred_aux4, pred_aux3, pred_aux2, _ = \
                self.unified_model(s_imgs, training=True)

            # Forward pass on clf batch → only need class output
            _, _, _, _, pred_cls = self.unified_model(c_imgs, training=True)

            # Segmentation losses
            loss_seg_primary = self.seg_loss_fn(s_masks, pred_mask)
            loss_aux4        = self.seg_loss_fn(s_masks, pred_aux4)
            loss_aux3        = self.seg_loss_fn(s_masks, pred_aux3)
            loss_aux2        = self.seg_loss_fn(s_masks, pred_aux2)

            # Deep supervision combined seg loss
            loss_seg = (loss_seg_primary
                        + AUX_W_D4 * loss_aux4
                        + AUX_W_D3 * loss_aux3
                        + AUX_W_D2 * loss_aux2)

            # Classification loss
            loss_clf = self.bce_loss(c_lbls, pred_cls)

            # Phase-aware weighted total
            total = self.seg_w * loss_seg + self.clf_w * loss_clf

            # Mixed precision scaling
            scaled = (self.optimizer.get_scaled_loss(total)
                      if hasattr(self.optimizer, 'get_scaled_loss') else total)

        grads = tape.gradient(scaled, self.unified_model.trainable_variables)
        if hasattr(self.optimizer, 'get_unscaled_gradients'):
            grads = self.optimizer.get_unscaled_gradients(grads)
        self.optimizer.apply_gradients(
            zip(grads, self.unified_model.trainable_variables))

        # Track primary seg loss only (cleaner signal for monitoring)
        self.m_loss_seg.update_state(loss_seg_primary)
        self.m_loss_clf.update_state(loss_clf)
        self.m_dice.update_state(dice_coef(s_masks, pred_mask))
        self.m_clf_acc.update_state(c_lbls, pred_cls)

    @tf.function
    def val_seg_step(self, s_imgs, s_masks):
        """Validation: segmentation branch only — runs over full val_seg_ds."""
        pred_mask, _, _, _, _ = self.unified_model(s_imgs, training=False)
        self.v_loss_seg.update_state(self.seg_loss_fn(s_masks, pred_mask))
        self.v_dice.update_state(dice_coef(s_masks, pred_mask))

    @tf.function
    def val_clf_step(self, c_imgs, c_lbls):
        """Validation: classification branch only — runs over full val_clf_ds."""
        _, _, _, _, pred_cls = self.unified_model(c_imgs, training=False)
        self.v_loss_clf.update_state(self.bce_loss(c_lbls, pred_cls))
        self.v_clf_acc.update_state(c_lbls, pred_cls)


# ─────────────────────────────────────────────────────────────────────────────
# 📅  LR SCHEDULE — Warmup + Cosine Decay with min_lr floor
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """
    Linear warmup → cosine decay → floor at min_lr.
    min_lr=5e-5 prevents the schedule from decaying to ~0 (which killed
    gradient signal by epoch 79 in the previous run).
    """
    def __init__(self, peak_lr=5e-4, warmup_steps=300,
                 total_steps=30000, min_lr=5e-5):
        super().__init__()
        self.peak_lr      = tf.cast(peak_lr,      tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.min_lr       = tf.cast(min_lr,        tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        # Linear warmup
        warmup = self.peak_lr * (step / (self.warmup_steps + 1e-8))
        # Cosine decay to min_lr
        cos    = self.min_lr + 0.5 * (self.peak_lr - self.min_lr) * (
            1.0 + tf.cos(np.pi *
                         (step - self.warmup_steps) /
                         (self.total_steps - self.warmup_steps + 1e-8))
        )
        return tf.maximum(
            tf.where(step < self.warmup_steps, warmup, cos),
            self.min_lr
        )

    def get_config(self):
        return {"peak_lr":      float(self.peak_lr.numpy()),
                "warmup_steps": float(self.warmup_steps.numpy()),
                "total_steps":  float(self.total_steps.numpy()),
                "min_lr":       float(self.min_lr.numpy())}


# ─────────────────────────────────────────────────────────────────────────────
# 🏃  TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
def train():
    print("\n⏳ Loading dataset paths...")
    seg_img_paths, seg_mask_paths = load_segmentation_paths(DATA_PATH)
    clf_img_paths, clf_labels     = load_classification_paths()

    s_tr_x, s_val_x, s_tr_y, s_val_y = train_test_split(
        seg_img_paths, seg_mask_paths, test_size=0.2, random_state=42)
    c_tr_x, c_val_x, c_tr_y, c_val_y = train_test_split(
        clf_img_paths, clf_labels, test_size=0.2, random_state=42)

    # Fixed step count — calculated once, never dynamic
    steps_per_epoch = min(len(s_tr_x) // BATCH_SIZE,
                          len(c_tr_x) // BATCH_SIZE)
    total_steps  = steps_per_epoch * EPOCHS
    warmup_steps = steps_per_epoch * WARMUP_EPOCHS
    print(f"  Steps/epoch: {steps_per_epoch} | Total: {total_steps} | Warmup: {warmup_steps}")

    print("⏳ Building tf.data pipelines...")
    train_seg_ds, train_clf_ds, val_seg_ds, val_clf_ds = build_datasets(
        s_tr_x, s_tr_y, s_val_x, s_val_y,
        c_tr_x, c_tr_y, c_val_x, c_val_y,
        steps_per_epoch
    )

    print("\n🏗️  Building Unified Model (v3)...")
    core_model = build_unified_model()
    core_model.summary(line_length=110)

    lr_schedule = WarmupCosineDecay(
        peak_lr=5e-4, warmup_steps=warmup_steps,
        total_steps=total_steps, min_lr=5e-5
    )
    optimizer = mixed_precision.LossScaleOptimizer(
        tf.keras.optimizers.Adam(learning_rate=lr_schedule)
    )

    trainer = UnifiedTrainer(core_model)
    trainer.compile(optimizer=optimizer, seg_loss_fn=combined_seg_loss)

    # Training state
    best_val_dice       = -np.inf
    patience            = 15
    patience_counter    = 0
    clf_converge_streak = 0
    phase2_active       = False

    history = {k: [] for k in [
        'loss_seg','loss_clf','dice','clf_acc',
        'val_loss_seg','val_loss_clf','val_dice','val_clf_acc',
        'seg_w','clf_w'
    ]}

    print(f"\n🚀 Training — up to {EPOCHS} epochs | patience={patience}")
    print(f"   Phase 1: seg_w={SEG_W_PHASE1} | clf_w={CLF_W_PHASE1}")
    print(f"   Phase 2 (auto after clf acc>{CLF_CONVERGE_ACC} × {CLF_CONVERGE_EPOCHS} epochs):"
          f" seg_w={SEG_W_PHASE2} | clf_w={CLF_W_PHASE2}")
    print(f"   Loss: 0.5×FocalTversky + 0.3×WeightedBCE + 0.2×Boundary")
    print(f"   DeepSupervision: aux_d4(w={AUX_W_D4}) aux_d3(w={AUX_W_D3}) aux_d2(w={AUX_W_D2})")
    print("=" * 72)

    for epoch in range(EPOCHS):
        # Reset train metrics
        for m in [trainer.m_loss_seg, trainer.m_loss_clf,
                  trainer.m_dice, trainer.m_clf_acc]:
            m.reset_state()

        seg_iter = iter(train_seg_ds)
        clf_iter = iter(train_clf_ds)

        phase_str = "P2" if phase2_active else "P1"
        print(f"\n[Epoch {epoch+1}/{EPOCHS}] [{phase_str}] "
              f"seg_w={trainer.seg_w.numpy():.2f}  clf_w={trainer.clf_w.numpy():.2f}")

        pb = tf.keras.utils.Progbar(steps_per_epoch, verbose=1)
        for step in range(steps_per_epoch):
            trainer.train_step(next(seg_iter), next(clf_iter))
            pb.update(step + 1, values=[
                ("LS",  trainer.m_loss_seg.result().numpy()),
                ("LC",  trainer.m_loss_clf.result().numpy()),
                ("Dc",  trainer.m_dice.result().numpy()),
                ("Acc", trainer.m_clf_acc.result().numpy()),
            ])

        # ── Validation — fully uncoupled, runs each dataset to completion ────
        for m in [trainer.v_loss_seg, trainer.v_loss_clf,
                  trainer.v_dice, trainer.v_clf_acc]:
            m.reset_state()

        for s_imgs, s_masks in val_seg_ds:
            trainer.val_seg_step(s_imgs, s_masks)

        for c_imgs, c_lbls in val_clf_ds:
            trainer.val_clf_step(c_imgs, c_lbls)

        v_ls  = trainer.v_loss_seg.result().numpy()
        v_lc  = trainer.v_loss_clf.result().numpy()
        v_d   = trainer.v_dice.result().numpy()
        v_acc = trainer.v_clf_acc.result().numpy()

        print(f"  → Val | Loss_Seg:{v_ls:.4f} | Loss_Clf:{v_lc:.4f} "
              f"| Dice:{v_d:.4f} | Acc:{v_acc:.4f}")

        # ── Phase switching ───────────────────────────────────────────────────
        if not phase2_active:
            clf_converge_streak = (clf_converge_streak + 1
                                   if v_acc >= CLF_CONVERGE_ACC else 0)
            if clf_converge_streak >= CLF_CONVERGE_EPOCHS:
                phase2_active = True
                trainer.seg_w.assign(SEG_W_PHASE2)
                trainer.clf_w.assign(CLF_W_PHASE2)
                print(f"\n  🔀 PHASE 2 ACTIVATED at epoch {epoch+1}  "
                      f"(clf acc={v_acc:.3f} ≥ {CLF_CONVERGE_ACC} × {CLF_CONVERGE_EPOCHS})")
                print(f"     Redirecting gradient budget: "
                      f"seg_w={SEG_W_PHASE2} | clf_w={CLF_W_PHASE2}\n")

        # ── History ───────────────────────────────────────────────────────────
        history['loss_seg'].append(trainer.m_loss_seg.result().numpy())
        history['loss_clf'].append(trainer.m_loss_clf.result().numpy())
        history['dice'].append(trainer.m_dice.result().numpy())
        history['clf_acc'].append(trainer.m_clf_acc.result().numpy())
        history['val_loss_seg'].append(v_ls)
        history['val_loss_clf'].append(v_lc)
        history['val_dice'].append(v_d)
        history['val_clf_acc'].append(v_acc)
        history['seg_w'].append(float(trainer.seg_w.numpy()))
        history['clf_w'].append(float(trainer.clf_w.numpy()))

        # ── Checkpoint + early stopping ───────────────────────────────────────
        if v_d > best_val_dice:
            best_val_dice    = v_d
            patience_counter = 0
            os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
            core_model.save(SAVE_PATH)
            print(f"  ✅ Best model saved (Val Dice: {best_val_dice:.4f})")
        else:
            patience_counter += 1
            print(f"  ⚠️  No improvement ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\n🛑 Early stopping at epoch {epoch+1}")
                break

        gc.collect()

    print(f"\n🏆 Training complete! Best Val Dice: {best_val_dice:.4f}")
    return core_model, history


# ─────────────────────────────────────────────────────────────────────────────
# 📈  PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
def plot_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Dream Polyp Unified Net v3 — Training History",
                 fontsize=15, fontweight='bold')

    pairs = [
        ('dice',     'val_dice',     'Dice Coefficient (↑ target ≥0.75)',  axes[0, 0]),
        ('clf_acc',  'val_clf_acc',  'Classification Accuracy',             axes[0, 1]),
        ('loss_seg', 'val_loss_seg', 'Segmentation Loss (FTL+WBCE+Bnd)',   axes[1, 0]),
        ('loss_clf', 'val_loss_clf', 'Classification Loss (BCE)',           axes[1, 1]),
    ]
    for tr_k, val_k, title, ax in pairs:
        ax.plot(history[tr_k],  label='Train', color='steelblue')
        ax.plot(history[val_k], label='Val',   color='tomato', linestyle='--')
        ax.set_title(title); ax.set_xlabel('Epoch')
        ax.legend(); ax.grid(alpha=0.3)

    # Phase weight schedule
    ax_w = axes[0, 2]
    ax_w.plot(history['seg_w'], label='seg_w', color='steelblue', linewidth=2)
    ax_w.plot(history['clf_w'], label='clf_w', color='tomato',    linewidth=2)
    ax_w.set_title('Phase-Aware Loss Weights'); ax_w.set_xlabel('Epoch')
    ax_w.set_ylim(-0.05, 1.05); ax_w.legend(); ax_w.grid(alpha=0.3)

    axes[1, 2].axis('off')
    plt.tight_layout()
    plt.savefig('training_history_v3.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Saved: training_history_v3.png")


# ─────────────────────────────────────────────────────────────────────────────
# 🔮  INFERENCE  ← output[0]=mask, output[4]=class (aux heads ignored)
# ─────────────────────────────────────────────────────────────────────────────
def infer(model, image_path):
    """
    Production inference. Model has 5 outputs [mask, aux4, aux3, aux2, class].
    Only outputs[0] and outputs[4] are used — auxiliary heads are ignored.
    """
    if not os.path.exists(image_path):
        print(f"❌ Image not found: {image_path}"); return

    raw  = tf.io.read_file(image_path)
    img  = tf.image.decode_image(raw, channels=3, expand_animations=False)
    orig = img.numpy().astype("uint8")
    img  = tf.cast(tf.image.resize(img, [IMG_SIZE, IMG_SIZE]), tf.float32) / 255.0
    inp  = tf.expand_dims(img, 0)

    outputs   = model.predict(inp, verbose=0)
    pred_mask = outputs[0]          # [1, 256, 256, 1]
    pred_prob = float(outputs[4][0][0])  # scalar

    mask_vis = (pred_mask[0, :, :, 0] > 0.5).astype(np.uint8) * 255
    label    = "POLYP DETECTED" if pred_prob >= 0.5 else "NON-POLYP"
    conf     = pred_prob * 100  if pred_prob >= 0.5 else (1 - pred_prob) * 100
    color    = 'limegreen'      if pred_prob >= 0.5 else 'tomato'

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(orig);                  axes[0].set_title("Input Image");      axes[0].axis("off")
    axes[1].imshow(mask_vis, cmap='gray'); axes[1].set_title("Predicted Mask");  axes[1].axis("off")

    overlay = orig.copy()
    mask_rs = tf.image.resize(
        mask_vis[..., np.newaxis], [orig.shape[0], orig.shape[1]], method='nearest'
    ).numpy().squeeze().astype(bool)
    overlay[mask_rs, 1] = np.clip(overlay[mask_rs, 1] + 80, 0, 255)
    axes[2].imshow(overlay); axes[2].set_title("Mask Overlay"); axes[2].axis("off")

    fig.suptitle(f"Diagnosis: {label}  ({conf:.1f}% confidence)",
                 color=color, fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f"\n{'='*50}\n  {label}  |  Confidence: {conf:.2f}%\n{'='*50}\n")
    return pred_mask, pred_prob


# ─────────────────────────────────────────────────────────────────────────────
# ▶  ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    core_model, history = train()
    plot_history(history)
    # infer(core_model, "/path/to/test_image.jpg")

✓ GPU Memory Growth Enabled — 1 GPU(s) found
✓ Mixed Precision (float16) enabled

⏳ Loading dataset paths...
  Segmentation samples: 1536
  Classification samples: pos=3762, neg=2520
  Steps/epoch: 307 | Total: 36840 | Warmup: 921
⏳ Building tf.data pipelines...

🏗️  Building Unified Model (v3)...


Model: "Dream_Polyp_Unified_Net_v3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                   ┃ Output Shape              ┃          Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)       │ (None, 256, 256, 3)       │                0 │ -                          │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ conv2d (Conv2D)                │ (None, 256, 256, 32)      │              864 │ input_image[0][0]          │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ batch_normalization            │ (None, 256, 256, 32)      │              128 │ conv2d[0][0]               │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)              │ (None, 256, 256, 32)      │               96 │ input_image[0][0]          │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ activation (Activation)        │ (None, 256, 256, 32)      │                0 │ batch_normalization[0][0]  │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ batch_normalization_1          │ (None, 256, 256, 32)      │              128 │ conv2d_1[0][0]             │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ add (Add)                      │ (None, 256, 256, 32)      │                0 │ activation[0][0],          │
│                                │                           │                  │ batch_normalization_1[0][… │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ global_average_pooling2d       │ (None, 32)                │                0 │ add[0][0]                  │
│ (GlobalAveragePooling2D)       │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ dense (Dense)                  │ (None, 4)                 │              128 │ global_average_pooling2d[… │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ dense_1 (Dense)                │ (None, 32)                │              128 │ dense[0][0]                │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ reshape (Reshape)              │ (None, 1, 1, 32)          │                0 │ dense_1[0][0]              │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ multiply (Multiply)            │ (None, 256, 256, 32)      │                0 │ add[0][0], reshape[0][0]   │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ separable_conv2d               │ (None, 256, 256, 8)       │              544 │ multiply[0][0]             │
│ (SeparableConv2D)              │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ separable_conv2d_1             │ (None, 256, 256, 8)       │            1,056 │ multiply[0][0]             │
│ (SeparableConv2D)              │                           │                  │                            │
├───

 Total params: 1,329,030 (5.07 MB)

 Trainable params: 1,325,444 (5.06 MB)

 Non-trainable params: 3,586 (14.01 KB)


🚀 Training — up to 120 epochs | patience=15
   Phase 1: seg_w=0.8 | clf_w=0.2
   Phase 2 (auto after clf acc>0.88 × 3 epochs): seg_w=0.97 | clf_w=0.03
   Loss: 0.5×FocalTversky + 0.3×WeightedBCE + 0.2×Boundary
   DeepSupervision: aux_d4(w=0.2) aux_d3(w=0.15) aux_d2(w=0.1)

[Epoch 1/120] [P1] seg_w=0.80  clf_w=0.20
